# Diwali Festival Sales Analysis - Phase 6 & 7: Machine Learning Model Development & Interpretation

**Objective**: Train and evaluate Linear Regression, Random Forest, and Gradient Boosting Regressors to predict customer purchase amounts (`Amount`) with scikit-learn pipelines.


In [ ]:
import os
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

data_path = os.path.join('..', 'data', 'processed', 'diwali_sales_cleaned.csv')
df = pd.read_csv(data_path)
print(f'Dataset ready for ML: {df.shape}')


### Feature Architecture & Pipeline Preprocessor
Features: Numerical (`Age`, `Orders`), Categorical (`Gender`, `Age Group`, `Marital_Status`, `State`, `Zone`, `Occupation`, `Product_Category`). Target: `Amount`.


In [ ]:
features_num = ['Age', 'Orders']
features_cat = ['Gender', 'Age Group', 'Marital_Status', 'State', 'Zone', 'Occupation', 'Product_Category']
target = 'Amount'

X = df[features_num + features_cat]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train set: {X_train.shape[0]} | Test set: {X_test.shape[0]}')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat)
    ]
)


### Model Training & Cross-Evaluation


In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    'Gradient Boosting Regressor': GradientBoostingRegressor(n_estimators=120, max_depth=5, learning_rate=0.09, random_state=42)
}

results = {}
trained_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', model)])
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    
    train_preds = pipe.predict(X_train)
    test_preds = pipe.predict(X_test)
    
    results[name] = {
        'Train MAE': mean_absolute_error(y_train, train_preds),
        'Train RMSE': root_mean_squared_error(y_train, train_preds),
        'Train R2': r2_score(y_train, train_preds),
        'Test MAE': mean_absolute_error(y_test, test_preds),
        'Test RMSE': root_mean_squared_error(y_test, test_preds),
        'Test R2': r2_score(y_test, test_preds)
    }

results_df = pd.DataFrame(results).T.round(4)
results_df


### Feature Importance & Interpretation
Examining which factors drive higher Diwali purchase amounts.


In [ ]:
best_pipe = trained_pipelines['Linear Regression']
fitted_prep = best_pipe.named_steps['preprocessor']
cat_names = fitted_prep.named_transformers_['cat'].get_feature_names_out(features_cat)
all_names = features_num + list(cat_names)
coefs = np.abs(best_pipe.named_steps['regressor'].coef_)

importance_df = pd.DataFrame({'Feature': all_names, 'Weight': coefs}).sort_values(by='Weight', ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(12), y='Feature', x='Weight', hue='Feature', palette='Oranges_r', legend=False)
plt.title('Top 12 Most Influential Features on Purchase Amount', fontsize=13, fontweight='bold')
plt.show()


### Persist Model Artifacts


In [ ]:
models_dir = os.path.join('..', 'models')
os.makedirs(models_dir, exist_ok=True)
joblib.dump(best_pipe, os.path.join(models_dir, 'best_model.pkl'))
print('Saved best_model.pkl successfully!')
